In [1]:
# v2版本是对之前模型的优化，主要体现在特征工程深化、加入新的特征、对现有特征进行分类、处理异常值和缺失值以及多期数据验证
#1. 导入和清洗数据

#1.1 导入建模所需要的库
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)

In [2]:
# 1.2 建立SQL 引擎, 导入数据

user = 'root'
password = '123456'
host = 'localhost'
port = 3306
database = 'sbux_analysis'

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4")
query = """
    SELECT 
        Number, Store_Name_cn, open_date, Tier,
        (CASE
            WHEN TZ = 'CBD' THEN 'keycity 大型综合商圈'
            WHEN TZ = 'C1' THEN '市级商业中心区'
            WHEN TZ = 'C2' THEN '区域级商业区'
            WHEN TZ = 'C3' THEN '社区型商业'
            WHEN TZ = 'O' THEN '办公商圈（写字楼、园区）'
            WHEN TZ = 'R' THEN '住宅'
            WHEN TZ = 'T' THEN '旅游'
            WHEN TZ = 'TS' THEN '交通枢纽'
            WHEN TZ = 'S' THEN '特殊商圈（学校、医院、博物馆）'
        END) AS channel,
        (CASE
            WHEN TA_Sub = 'Airport-Arrival Mix' THEN '机场'
            WHEN TA_Sub = 'Airport-Departure Mix' THEN '机场'
            WHEN TA_Sub = 'Airport-Domestic' THEN '机场'
            WHEN TA_Sub = 'Airport-GTC' THEN '机场'
            WHEN TA_Sub = 'Airport-International' THEN '机场'
            WHEN TA_Sub = 'Campus' THEN '各类办公园区'
            WHEN TA_Sub = 'Cinema' THEN '电影院'
            WHEN TA_Sub = 'C-Street' THEN '商业街'
            WHEN TA_Sub = 'Culture' THEN '书店/图书馆等文化场所'
            WHEN TA_Sub = 'Dept.' THEN '百货'
            WHEN TA_Sub = 'Headquarter' THEN '企业总部'
            WHEN TA_Sub = 'Headquarter-Internal' THEN '企业总部-内部'
            WHEN TA_Sub = 'Highway' THEN '高速公路服务区'
            WHEN TA_Sub = 'Hospital' THEN '医院'
            WHEN TA_Sub = 'Hotel' THEN '酒店'
            WHEN TA_Sub = 'Lux' THEN '奢侈品商场'
            WHEN TA_Sub = 'Mall' THEN '购物中心'
            WHEN TA_Sub = 'Mart' THEN '超市/大卖场'
            WHEN TA_Sub = 'Metro' THEN '地铁'
            WHEN TA_Sub = 'Neighborhood' THEN '社区-商业中心'
            WHEN TA_Sub = 'Office' THEN '写字楼门店'
            WHEN TA_Sub = 'O-Street' THEN '办公-餐饮街'
            WHEN TA_Sub = 'Others' & TA = 'C' THEN '其他商业类型'
            WHEN TA_Sub = 'Others' & TA = 'O' THEN '其他办公类型'
            WHEN TA_Sub = 'Others' & TA = 'R' THEN '其他类型社区店'
            WHEN TA_Sub = 'Others' & TA = 'T' THEN '其他旅游类型'
            WHEN TA_Sub = 'Others' & TA = 'TS' THEN '其他类型交通枢纽'
            WHEN TA_Sub = 'Others' & TA = 'Others' THEN '其他特殊类型'
            WHEN TA_Sub = 'Outlets' THEN '奥莱'
            WHEN TA_Sub = 'Passby' THEN '街铺'
            WHEN TA_Sub = 'Railway-Arrival' THEN '火车站'
            WHEN TA_Sub = 'Railway-inside' THEN '火车站'
            WHEN TA_Sub = 'Railway-outside' THEN '火车站'
            WHEN TA_Sub = 'REBA' THEN '餐饮/酒吧街'
            WHEN TA_Sub = 'Residential' THEN '服务社区的商区'
            WHEN TA_Sub = 'Special Market' THEN '专业市场'
            WHEN TA_Sub = 'T' THEN '独立性较强的景点，一般需购票进入'
            WHEN TA_Sub = 'Theater' THEN '剧院/音乐厅'
            WHEN TA_Sub = 'T-Street' THEN '旅游特色商业街'
            WHEN TA_Sub = 'University' THEN '大学'
        END) AS channel_sub,
        Region, _area AS area, 
        FY25_P08_ADT, FY25_P09_ADT, FY25_P10_ADT, FY25_P11_ADT, FY25_P12_ADT, FY26_P01_ADT, FY23_ADT, FY24_ADT, FY25_ADT, FY26_YTD_ADT,
        FY25_P08_AT, FY25_P09_AT, FY25_P10_AT, FY25_P11_AT, FY25_P12_AT, FY26_P01_AT, FY23_AT, FY24_AT, FY25_AT, FY26_YTD_AT,
        FY25_P08_netrevenue, FY25_P09_netrevenue, FY25_P10_netrevenue, FY25_P11_netrevenue, FY25_P12_netrevenue, FY26_P01_netrevenue,  FY23_netrevenue,
        FY24_netrevenue, FY25_netrevenue, FY26_YTD_netrevenue,
        FY25_P08_SPC, FY25_P09_SPC, FY25_P10_SPC, FY25_P11_SPC, FY25_P12_SPC, FY26_P01_SPC, FY23_SPC, FY24_SPC, FY25_SPC, FY26_YTD_SPC,
        FY25_P08_rent, FY25_P09_rent, FY25_P10_rent, FY25_P11_rent, FY25_P12_rent, FY26_P01_rent, FY23_rent, FY24_rent, FY25_rent, FY26_YTD_rent
   FROM sbux_store
"""
df = pd.read_sql(query,engine)

In [3]:
# 1.3 清洗数据

cols_to_clean = ['area','FY25_P08_ADT', 'FY25_P09_ADT', 'FY25_P10_ADT', 'FY25_P11_ADT', 
                 'FY25_P12_ADT', 'FY26_P01_ADT', 'FY23_ADT', 'FY24_ADT', 'FY25_ADT', 'FY26_YTD_ADT',
                 'FY25_P08_AT', 'FY25_P09_AT', 'FY25_P10_AT', 'FY25_P11_AT', 
                 'FY25_P12_AT', 'FY26_P01_AT', 'FY23_AT', 'FY24_AT', 'FY25_AT', 'FY26_YTD_AT',
                 'FY25_P08_rent', 'FY25_P09_rent', 'FY25_P10_rent', 'FY25_P11_rent', 
                 'FY25_P12_rent', 'FY26_P01_rent', 'FY23_rent', 'FY24_rent', 'FY25_rent', 'FY26_YTD_rent',
                 'FY25_P08_netrevenue', 'FY25_P09_netrevenue', 'FY25_P10_netrevenue', 'FY25_P11_netrevenue', 
                 'FY25_P12_netrevenue', 'FY26_P01_netrevenue', 'FY23_netrevenue', 'FY24_netrevenue', 'FY25_netrevenue', 'FY26_YTD_netrevenue',
                 'FY25_P08_SPC', 'FY25_P09_SPC', 'FY25_P10_SPC', 'FY25_P11_SPC', 
                 'FY25_P12_SPC', 'FY26_P01_SPC', 'FY23_SPC', 'FY24_SPC', 'FY25_SPC', 'FY26_YTD_SPC']
# 清除千分位符号
for i in cols_to_clean:
    df[i] = df[i].astype(str)
    df[i] = df[i].str.replace(',', '', regex=False)
    df[i] = pd.to_numeric(df[i], errors='coerce')

In [4]:
df['open_date'] = pd.to_datetime(df['open_date'])

#筛选截止FY26 即2025/9/29之前开业已满六个月的门店，业绩稳定门店
fy26_startdate = pd.Timestamp('2025/9/29')

#计算到2025/9/29 之前的月数
df['months_since_opened'] = (fy26_startdate.year - df['open_date'].dt.year)*12 + (fy26_startdate.month - df['open_date'].dt.month)
df_stable_store = df[df['months_since_opened'] >= 6].copy()

In [5]:
df_model = df_stable_store.copy()
df_model['rent_per_square'] = (df_model['FY25_P12_rent'] / df_model['area']).round(2)

# 1.4 选择特征列
features = ['Tier', 'channel', 'channel_sub', 'area', 'FY25_P12_rent', 'rent_per_square']
target = 'FY26_P01_ADT'

#去除缺失值
df_clean = df_model[features + [target]].dropna()
print(f"原始样本量:{len(df_model)}, 清洗后:{len(df_clean)}")

原始样本量:7641, 清洗后:7230


In [6]:
#2.划分训练集和测试集

x = df_clean[features]
y = df_clean[target]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=42) # 80% 的数据训练模型，20% 的数据测试效果

In [7]:
#3. 搭建数据处理流水线 + 线性回归
# Tier, Channel, channel_sub 三个字段的数据类型是文本，需要转换为数字

categorical_features = ['Tier', 'channel', 'channel_sub']
numeric_features = ['area', 'FY25_P12_rent', 'rent_per_square']

# 创建预处理步骤，处理文本字段
preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numeric_features),
            ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
        ])
# 线性回归
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

#训练
lr_model.fit(x_train, y_train)

#预测与评估
y_pred_lr = lr_model.predict(x_test)
r2_lr = r2_score(y_test, y_pred_lr)
print(f"线性回归 R²:{r2_lr:.4f}")

线性回归 R²:0.5550


In [8]:
#4. 随机森林回归，随机森林可以自动处理非线性关系，且不需要太复杂的预处理（类别特征需要编码，上一步已经完成了类别独热编码）

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_model.fit(x_train, y_train)
y_pred_rf = rf_model.predict(x_test)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"随机森林 R²: {r2_rf:.4f}")

随机森林 R²: 0.5901


In [9]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, verbosity=0))
])

# lightGBM
lgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LGBMRegressor(n_estimators=100, learning_rate=0.1, num_leaves=31, random_state=42, verbosity=-1))
])

xgb_model.fit(x_train, y_train)
print(f"XGBoost R²:" , xgb_model.score(x_test, y_test))

lgb_model.fit(x_train, y_train)
print(f"LightGBM R²:", lgb_model.score(x_test, y_test))

XGBoost R²: 0.5895904889031471
LightGBM R²: 0.6122991795867103


C:\Users\86137\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [10]:
#5 保存模型
# 随机森林的R²大于线性回归的R²，所以选择保留随机森林模型

joblib.dump(lgb_model, 'adt_predictor_rf.pkl')
print("模型已经保存为adt_predictor_rf.pkl")

模型已经保存为adt_predictor_rf.pkl
